<a href="https://colab.research.google.com/github/ThousandAI/AI-DeepLearning/blob/main/FFN_vs_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FFN vs CNN:MNIST 與 CIFAR-10 比較實驗

我們要問一個很簡單的問題:

> **只用全連接層(FFN)夠不夠?什麼時候一定要用 CNN?**

做法:同樣的訓練條件,只換模型和資料集,跑 4 組實驗。

| | MNIST(手寫數字) | CIFAR-10(彩色照片) |
|---|---|---|
| **FFN** | 第 1 組 | 第 2 組 |
| **CNN** | 第 3 組 | 第 4 組 |

執行前記得開 GPU:上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**


## 1. 準備

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 有 GPU 就用 GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("使用:", device)

torch.manual_seed(0)   # 固定隨機種子,讓每次跑的結果差不多

使用: cuda


## 2. 下載資料

MNIST 是 28x28 的黑白手寫數字,CIFAR-10 是 32x32 的彩色照片(飛機、貓、狗...),兩個都是 10 類。

`Normalize` 是把像素值調整到 0 附近,這樣訓練比較穩定。

In [ ]:
def load_data(name):
    if name == "mnist":
        tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        train = datasets.MNIST("./data", train=True,  download=True, transform=tf)
        test  = datasets.MNIST("./data", train=False, download=True, transform=tf)
    else:
        tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
        train = datasets.CIFAR10("./data", train=True,  download=True, transform=tf)
        test  = datasets.CIFAR10("./data", train=False, download=True, transform=tf)

    train_loader = DataLoader(train, batch_size=128, shuffle=True)
    test_loader  = DataLoader(test,  batch_size=256, shuffle=False)
    return train_loader, test_loader


mnist_train, mnist_test = load_data("mnist")
cifar_train, cifar_test = load_data("cifar10")
print("資料準備好了")

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 494kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.50MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.4MB/s]
  2%|▏         | 4.23M/170M [00:36<24:20, 114kB/s]

### 先看看資料長什麼樣子

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4))

# 第一排:MNIST
images, labels = next(iter(mnist_train))
for i in range(6):
    axes[0][i].imshow(images[i][0], cmap="gray")
    axes[0][i].set_title(f"MNIST: {labels[i].item()}")
    axes[0][i].axis("off")

# 第二排:CIFAR-10
classes = ["飛機","汽車","鳥","貓","鹿","狗","蛙","馬","船","卡車"]
images, labels = next(iter(cifar_train))
for i in range(6):
    axes[1][i].imshow(images[i].permute(1, 2, 0) * 0.5 + 0.5)  # 還原成可以看的顏色
    axes[1][i].set_title(f"CIFAR: {classes[labels[i]]}")
    axes[1][i].axis("off")

plt.tight_layout()
plt.show()

## 3. 兩個模型

### FFN(全連接網路)

第一步 `Flatten` 把整張圖攤平成一長條數字。

MNIST:`28 x 28 = 784` 個數字
CIFAR:`3 x 32 x 32 = 3072` 個數字

**注意:攤平之後,「哪些像素原本是鄰居」這件事就完全消失了。** 這是等一下要觀察的重點。

In [ ]:
def build_ffn(channels, size):
    input_size = channels * size * size

    return nn.Sequential(
        nn.Flatten(),                  # 攤平成一長條
        nn.Linear(input_size, 256),    # 全連接層
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Linear(128, 10)             # 輸出 10 類
    )

### CNN(卷積網路)

`Conv2d` 一次只看一小塊(3x3)的區域,所以它知道哪些像素是鄰居。
`MaxPool2d(2)` 把圖縮小一半,讓後面的層可以看到更大的範圍。

圖片尺寸的變化:`32 → 16 → 8`(MNIST 是 `28 → 14 → 7`),所以最後攤平的長度是 `64 x (size/4) x (size/4)`。

In [ ]:
def build_cnn(channels, size):
    final_size = size // 4    # 經過兩次 MaxPool,長寬各縮小 4 倍

    return nn.Sequential(
        nn.Conv2d(channels, 32, kernel_size=3, padding=1),   # 看 3x3 的小區域
        nn.ReLU(),
        nn.MaxPool2d(2),                                     # 圖片縮一半

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),                                     # 再縮一半

        nn.Flatten(),
        nn.Linear(64 * final_size * final_size, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

## 4. 訓練的程式

這段是通用的,4 組實驗都用同一個函式,才叫做公平比較。

In [ ]:
def train(model, train_loader, test_loader, epochs=5):
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    train_acc_list = []
    test_acc_list = []

    for epoch in range(epochs):
        # ---- 訓練 ----
        model.train()
        correct = 0
        total = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            pred = model(x)              # 前向:算出預測
            loss = loss_fn(pred, y)      # 算出錯多少

            optimizer.zero_grad()        # 清掉上一輪的梯度
            loss.backward()              # 反向傳播
            optimizer.step()             # 更新參數

            correct += (pred.argmax(1) == y).sum().item()
            total += y.size(0)
        train_acc = correct / total

        # ---- 測試(不更新參數)----
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                correct += (pred.argmax(1) == y).sum().item()
                total += y.size(0)
        test_acc = correct / total

        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print(f"  第 {epoch+1} 輪:訓練 {train_acc*100:.2f}%  測試 {test_acc*100:.2f}%")

    return train_acc_list, test_acc_list


def count_params(model):
    return sum(p.numel() for p in model.parameters())

## 5. 開始跑 4 組實驗

在 T4 GPU 上大約 3~5 分鐘。

In [ ]:
EPOCHS = 5
results = {}

experiments = [
    ("MNIST + FFN", build_ffn(1, 28), mnist_train, mnist_test),
    ("MNIST + CNN", build_cnn(1, 28), mnist_train, mnist_test),
    ("CIFAR + FFN", build_ffn(3, 32), cifar_train, cifar_test),
    ("CIFAR + CNN", build_cnn(3, 32), cifar_train, cifar_test),
]

for name, model, tr, te in experiments:
    print(f"\n===== {name}(參數量 {count_params(model):,})=====")
    train_acc, test_acc = train(model, tr, te, epochs=EPOCHS)
    results[name] = {
        "train": train_acc,
        "test": test_acc,
        "params": count_params(model)
    }

print("\n跑完了!")

## 6. 結果表格

In [ ]:
print(f"{'實驗':<14}{'參數量':>12}{'訓練正確率':>12}{'測試正確率':>12}")
print("-" * 52)
for name, r in results.items():
    print(f"{name:<14}{r['params']:>12,}{r['train'][-1]*100:>11.2f}%{r['test'][-1]*100:>11.2f}%")

## 7. 畫成圖

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左圖:MNIST
for name in ["MNIST + FFN", "MNIST + CNN"]:
    acc = [a * 100 for a in results[name]["test"]]
    axes[0].plot(range(1, EPOCHS + 1), acc, marker="o", label=name)
axes[0].set_title("MNIST:兩者差不多")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("測試正確率 (%)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 右圖:CIFAR-10
for name in ["CIFAR + FFN", "CIFAR + CNN"]:
    acc = [a * 100 for a in results[name]["test"]]
    axes[1].plot(range(1, EPOCHS + 1), acc, marker="o", label=name)
axes[1].set_title("CIFAR-10:差很多")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("測試正確率 (%)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 結論

跑出來的數字大概會是這樣:

| 實驗 | 測試正確率 |
|---|---|
| MNIST + FFN | 約 97% |
| MNIST + CNN | 約 99% |
| CIFAR + FFN | 約 **45~50%** |
| CIFAR + CNN | 約 **68~72%** |

### 三個重點

**1. 在 MNIST 上,FFN 其實表現得很好**

只差 CNN 大約 2%。因為手寫數字已經被整理得很乾淨了 — 黑白、置中、大小一致。攤平雖然弄丟了空間資訊,但影響不大。

**如果只看 MNIST,你會以為模型架構不重要。** 這是很常見的誤解。

**2. 換到 CIFAR-10,差距立刻變成 20% 以上**

真實照片裡,貓可能在左邊也可能在右邊、可能大可能小,而且要靠毛的紋理、輪廓來辨認。這些都是「一小塊區域裡的圖案」,正好是卷積在做的事。FFN 攤平之後看不到這些,所以學不起來。

**3. 這不是參數多寡的問題**

看一下表格 — CIFAR 的 FFN 參數量比 CNN 還多,但正確率低很多。所以問題不在「模型不夠大」,而在「模型的結構適不適合這種資料」。

---

### 可以讓學生自己試的

1. 把 `EPOCHS` 改成 15,看 CIFAR + FFN 會不會追上來(答案:不會,而且會開始「背答案」— 訓練正確率一直升,測試正確率停住)
2. 把 FFN 的 256 改成 2048,參數變超多,正確率會提升嗎?
3. 在 CNN 裡再加一組 `Conv2d + ReLU + MaxPool2d`,會更好嗎?
4. 把 CIFAR 圖片轉成黑白再訓練,結果會怎樣?
